# CI-Triage-Env — Evaluation & Ablation Notebook

Colab-runnable notebook for judges to reproduce all results.

Steps:
1. Install dependencies
2. Load trained checkpoint from HF Hub
3. Run full 5-baseline evaluation
4. Generate all metric plots
5. Run reward-layer ablations (optional, GPU, ~5h)
6. Populate README with results

**Prerequisites**: `HF_TOKEN`, `WANDB_API_KEY` as Colab secrets.

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q unsloth trl transformers accelerate peft
!pip install -q wandb datasets huggingface_hub openai httpx fastapi uvicorn pydantic jsonschema
!pip install -q matplotlib seaborn pandas tabulate
!pip install -q -e .  # install ci_triage_env package

In [ ]:
# Cell 2: Environment setup
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['WANDB_PROJECT'] = 'ci-triage-env'

# Config — replace with your values
HF_DATASET_REPO = 'YOUR_ORG/ci-triage-scenarios'
HF_MODEL_REPO   = 'YOUR_ORG/ci-triage-trained-qwen3.5-4b'
WANDB_RUN_ID    = 'YOUR_WANDB_RUN_ID'  # e.g. 'entity/ci-triage-env/abc123'

In [ ]:
# Cell 3: Download scenario corpus and trained checkpoint
from huggingface_hub import snapshot_download

scen_dir = snapshot_download(
    HF_DATASET_REPO, repo_type='dataset',
    local_dir='data_artifacts/scenarios'
)
ckpt_dir = snapshot_download(
    HF_MODEL_REPO, repo_type='model',
    local_dir='checkpoints/grpo_full'
)
print(f'Scenarios: {scen_dir}')
print(f'Checkpoint: {ckpt_dir}')

In [ ]:
# Cell 4: Start env server in background
import subprocess, time
server_proc = subprocess.Popen(
    ['python', '-m', 'ci_triage_env.env.server'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(4)
print('Env server started, PID:', server_proc.pid)

In [ ]:
# Cell 5: Run full 5-baseline evaluation
from pathlib import Path
import pandas as pd
from ci_triage_env.training.eval import Evaluator
from ci_triage_env.training.plotting import plot_all_eval_metrics

evaluator = Evaluator(
    eval_set_path='data_artifacts/scenarios/held_out/',
    trained_checkpoint='checkpoints/grpo_full/',
)
df_eval = evaluator.run_all(seeds=[1, 2, 3])

out = Path('data_artifacts/results/')
out.mkdir(parents=True, exist_ok=True)
df_eval.to_csv(out / 'eval.csv', index=False)

print(df_eval.groupby('baseline').agg({
    'diagnosis_correct': 'mean',
    'total_reward': 'mean',
    'tool_call_count': 'mean',
}))

plot_all_eval_metrics(df_eval, out / 'plots/')
print('Plots saved to data_artifacts/results/plots/')

In [ ]:
# Cell 6: Pull training curves from W&B
from ci_triage_env.training.curves import plot_training_curves_from_wandb

plot_training_curves_from_wandb(
    run_id=WANDB_RUN_ID,
    output_dir=Path('data_artifacts/results/plots/'),
)
print('Training curves saved.')

In [ ]:
# Cell 7: Run reward-layer ablations (~5h on A100; set RUN_ABLATIONS=True to enable)
RUN_ABLATIONS = False

if RUN_ABLATIONS:
    from ci_triage_env.training.ablations import ABLATIONS, run_ablation
    from ci_triage_env.training.curves import plot_ablation_summary

    abl_results = []
    for name, overrides in ABLATIONS.items():
        print(f'=== Ablation: {name} ===')
        df_abl = run_ablation(name, overrides, total_steps=1000)
        abl_results.append(df_abl)
        print(df_abl.groupby('baseline')['diagnosis_correct'].mean())

    df_full_abl = pd.concat(abl_results, ignore_index=True)
    df_full_abl.to_csv(out / 'ablations.csv', index=False)
    plot_ablation_summary(df_full_abl, output_dir=out / 'plots/')
    print('Ablations saved.')
else:
    print('Ablations skipped (set RUN_ABLATIONS=True to run).')

In [ ]:
# Cell 8: Populate README with results
from ci_triage_env.training.finalize_readme import populate_readme

n = populate_readme(
    eval_csv=out / 'eval.csv',
    ablation_csv=out / 'ablations.csv',
    plots_dir=out / 'plots/',
)
print(f'Replaced {n} markers in README.md')

# Check for any remaining unfilled markers
import subprocess
result = subprocess.run(['grep', '-c', r'\[FILL', 'README.md'], capture_output=True, text=True)
remaining = int(result.stdout.strip() or 0)
if remaining:
    print(f'WARNING: {remaining} unfilled [FILL] marker(s) remain in README.md')
else:
    print('README.md is clean — no unfilled markers.')